# Get preds on every hyper-param combination across folds/repeats

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import ALPHA_LIST
from src.swarm import run_swarms, make_reduced, make_full_lr, make_full_shot

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-8B-Instruct",
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = ["self", "parent"]
PROB_AGG_TYPES = [
    "reduced",
    "block",
    "max",
    "avg",
    "0_shot",
]
BASE_SAVE_DIR = BASE_PATH / "initial_preds"

## Get model preds at every:

- (Self, Parent) x Alpha Val X L1 Val x Repeat (1-10) x test fold combination for REDUCED
- (Self, Parent) x (block, avg, max) x (8B, 70B) x (turns, pt_noshort) x Alpha Val X L1 Val x Repeat (1-10) x test fold combination for FULL

In [ ]:
N_JOBS = 4
N_THREADS = 2

In [ ]:
CMD_DIR = BASE_PATH / "swarm_dir" / "commands"
full_cmd_list = []
shot_pred_types = [
    "proba-limited",
    # "proba",
    # "binary",
]
shot_desc_types = [
    "original",
    "augment",
]
for agg_type in PROB_AGG_TYPES:
    for impact_type in IMPACT_TYPES:
        swarm_path = CMD_DIR / agg_type / f"{impact_type}.swarm"
        swarm_path.parent.mkdir(exist_ok=True, parents=True)
        if swarm_path.exists():
            swarm_path.unlink()
        cmd_list = []
        for alpha_val in ALPHA_LIST:
            ## Reduced model
            if agg_type == "reduced":
                cmd_str = make_reduced(
                    impact_type=impact_type,
                    alpha_val=alpha_val,
                    base_save_dir=BASE_SAVE_DIR,
                    n_jobs=N_JOBS,
                    n_threads=N_THREADS,
                )
                cmd_list.append(cmd_str)
                continue
            ## Full model (block, max, avg)
            for model_name in MODEL_NAMES:
                if agg_type in ["block", "max", "avg"]:
                    lr_cmd_list = make_full_lr(
                        seg_types=SEG_TYPES,
                        model_name=model_name,
                        impact_type=impact_type,
                        alpha_val=alpha_val,
                        agg_type=agg_type,
                        base_save_dir=BASE_SAVE_DIR,
                        n_jobs=N_JOBS,
                        n_threads=N_THREADS,
                    )
                    cmd_list += lr_cmd_list
                ## 0 shot
                else:
                    for pred_type in shot_pred_types:
                        for desc_type in shot_desc_types:
                            cmd_str = make_full_shot(
                                model_name=model_name,
                                impact_type=impact_type,
                                alpha_val=alpha_val,
                                temp=0.6,
                                pred_type=pred_type,
                                desc_type=desc_type,
                                base_save_dir=BASE_SAVE_DIR,
                                n_jobs=N_JOBS,
                                n_threads=N_THREADS,
                            )
                            cmd_list.append(cmd_str)
        swarm_path.write_text("\n".join(cmd_list))
        full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run sequentially locally

In [ ]:
# import subprocess

# BASE_SAVE_DIR.mkdir(exist_ok=True, parents=True)
# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     subprocess.run(cmd, shell=True, check=True)

Run swarm

In [ ]:
run_swarms(
    agg_list=PROB_AGG_TYPES,
    impact_list=["parent", "self"],
    swarm_log_dir=BASE_PATH / "swarm_dir" / "logs",
    cmd_dir=CMD_DIR,
    n_jobs=N_JOBS,
    n_threads=N_THREADS,
)